zipcode = (10003.0   , 54
10013.0   ,  37
11372.0   ,  37
11354.0   ,  37
10016.0   , 35)

I now need to find the 5 sensor ID of the sensors located in those areas. One sensor : 172823 for the 10013 zip code, Jackson Heights (zip code : 11372) 295845,

Sensor ID located at Washington Square Village 189249 for Zip Code 10012, sensor ID for hells kitchen (10019) 29183

In [ ]:
import os
import time
import requests
import pandas as pd
from io import StringIO

API_KEY = "YOUR API_KEY"
SENSOR_IDS = [172823, 295845, 189249, 29183, 150174, 285560, 306402, 90247, 319136, 184329]  # all my sensors
restaurant_ids = [172823, 295845, 189249, 29183, 150174]
no_restaurant_ids = [285560, 306402, 90247, 319136, 184329]

# build the lookup dictionary
labels = {}
for x in restaurant_ids:
    labels[x] = "restaurant"
for x in no_restaurant_ids:
    labels[x] = "no restaurant"


# 14-day windows from June 1 to Sept 30
starts = pd.date_range("2026-06-01", "2026-09-30", freq="14D", tz="UTC")
ends = starts[1:].append(pd.DatetimeIndex([pd.Timestamp("2026-09-30", tz="UTC")]))

os.makedirs("../output/sensor_data", exist_ok=True)


for sid in labels:
    out_file = f"sensor_data/{sid}.csv"
    if os.path.exists(out_file):
        continue    

frames = []
for sid in SENSOR_IDS:
    for t0, t1 in zip(starts, ends):
        r = requests.get(
            f"https://api.purpleair.com/v1/sensors/{sid}/history/csv",
            headers={"X-API-Key": API_KEY},
            params={
                "start_timestamp": t0.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "end_timestamp": t1.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "average": 60,
                "fields": "pm2.5_atm,pm2.5_cf_1",
            },
            timeout=60,
        )
        if r.status_code == 200:
            df = pd.read_csv(StringIO(r.text))
            df["sensor_index"] = sid
            frames.append(df)
        else:
            print(sid, t0.date(), r.status_code, r.text[:200])
        time.sleep(1)

all_df = pd.concat(frames, ignore_index=True)
all_df["zone"] = all_df["sensor_index"].map(labels) 
all_df.to_csv("../output/air_pollution.csv", index=False)
print(all_df.shape)

(27096, 5)


In [13]:
df_pollution = pd.read_csv("air_pollution.csv")
print(df_pollution)

                 time_stamp  sensor_index  pm2.5_atm  pm2.5_cf_1  \
0      2026-06-02T23:00:00Z        172823        4.8         4.8   
1      2026-06-02T17:00:00Z        172823        5.2         5.2   
2      2026-06-02T14:00:00Z        172823        4.3         4.3   
3      2026-06-02T06:00:00Z        172823        6.1         6.1   
4      2026-06-01T20:00:00Z        172823        4.9         4.9   
...                     ...           ...        ...         ...   
27091  2026-09-28T08:00:00Z        184329        1.3         1.3   
27092  2026-09-28T02:00:00Z        184329        0.4         0.4   
27093  2026-09-28T23:00:00Z        184329        1.7         1.7   
27094  2026-09-28T19:00:00Z        184329        1.5         1.5   
27095  2026-09-28T11:00:00Z        184329        1.5         1.5   

                zone  
0         restaurant  
1         restaurant  
2         restaurant  
3         restaurant  
4         restaurant  
...              ...  
27091  no restaurant  

In [14]:
df_pollution.sample(10)

,time_stamp,sensor_index,pm2.5_atm,pm2.5_cf_1,zone
15914,2026-07-27T01:00:00Z,285560,11.8,11.8,no restaurant
12129,2026-06-16T11:00:00Z,150174,3.0,3.2,restaurant
9485,2026-07-09T03:00:00Z,29183,6.4,6.4,restaurant
7460,2026-07-28T06:00:00Z,189249,18.9,18.9,restaurant
6697,2026-07-09T20:00:00Z,189249,7.2,7.3,restaurant
20510,2026-06-27T14:00:00Z,90247,11.6,11.6,no restaurant
22388,2026-08-29T21:00:00Z,90247,2.0,2.0,no restaurant
7354,2026-08-09T08:00:00Z,189249,13.3,13.3,restaurant
4076,2026-07-15T23:00:00Z,295845,135.3,204.1,restaurant
24568,2026-06-23T08:00:00Z,184329,7.5,7.5,no restaurant


I think there is a way to not run the requests every time but I don't know how to do it. 